# Exercises (Student) - MCP Client with LLM

In [21]:
!pip install -q mcp nest_asyncio requests azure-ai-inference azure-core

In [14]:

import os
from pathlib import Path
MCP_HTTP_TOKEN = os.getenv("MCP_HTTP_TOKEN", "devtoken123")
USE_REAL_LLM = True  # flip True if GITHUB_TOKEN is set


In [3]:
import asyncio
import nest_asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()


In [4]:
%%writefile server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("DemoServer")

#To-Do: Add decorator to register add as a tool
@mcp.tool()
def add(a: int, b: int) -> int:
    "Add two numbers."
    return a + b

# TODO (optional exercise): add multiply(a, b) here if you want an extra tool
@mcp.tool()
def multiply(a: float, b: float) -> float:
    "Multiple two numbers"
    return a * b



#To-Do: Add decorator to register greet as a tool
@mcp.tool()
def greet(name: str) -> str:
    "Return a greeting string."
    return f"Hey, {name}!"

if __name__ == "__main__":
    mcp.run()


Overwriting server.py


## Exercise 1 (provide answer)

#To-Do: Why is STDIO transport simple for local MCP dev compared to HTTP?

STDIO is simpler because it doesn't require the network layer, no port, no url, no auth, the client just spawns the server as a subprocess and talk to it over its stdin and stdout pipeline.  But HTTP is through the network, the server is separated from client and has to be started independently.

trade off is that stdio is only for client and server on the same machine, it can't be shared across a network or reached by multiple clients at once.


## Exercise 2

In [5]:

import asyncio
import nest_asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()

async def ex2_connect():
    params = StdioServerParameters(command="mcp", args=["run", "server.py"]) #To-Do: Create StdioServerParameters with command to run server.py
    async with stdio_client(params, errlog=True) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()


In [6]:
# In a new cell
await ex2_connect()
print("Exercise 2: OK (connected and initialized)")


Exercise 2: OK (connected and initialized)


## Exercise 3

In [7]:
async def ex3_list():
    params = StdioServerParameters(command="mcp", args=["run", "server.py"]) 
    async with stdio_client(params, errlog=True) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()
            resources = await session.list_resources()
            print("RESOURCES:", resources)
            tools = await session.list_tools()
            for t in tools.tools:
                print(t.name, t.inputSchema.get("properties", {}))


In [8]:
await ex3_list()

[07/22/26 21:21:49] INFO     Processing request of type            server.py:733
                             ListResourcesRequest                               
RESOURCES: meta=None nextCursor=None resources=[]
                    INFO     Processing request of type            server.py:733
                             ListToolsRequest                                   
add {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}
multiply {'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}
greet {'name': {'title': 'Name', 'type': 'string'}}


## Exercise 4

#To-Do: Explain how the conversion to llm tool happens in MCP server code ?

it's basically a remapping of mcp tool object to a json format

In [9]:

def convert_to_llm_tool(tool):
    return {
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description or "mcp tool",
            "parameters": {
                "type": "object",
                "properties": tool.inputSchema.get("properties", {}),
                "required": tool.inputSchema.get("required", []),
            },
        },
    }


## Exercise 5

**Plan & execute:** Use stub (or real) LLM to propose `tool_calls`, then execute them and print results for a prompt like “Add 2 to 20.”

In [ ]:

import asyncio
import json
import nest_asyncio
from typing import Any, Dict, List
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
import re
nest_asyncio.apply()

def stub_plan(prompt: str, functions: list[Dict[str, Any]]):
    """Fake LLM planner: pattern match simple prompts to tool calls"""
    numbers = [int(n) for n in re.findall(r"-?\d+", prompt)]
    if "add" in prompt.lower() and len(numbers) >= 2:
        return [{"name": "add", "args": {"a": numbers[0], "b": numbers[1]}}]
    if "multiply" in prompt.lower() and len(numbers) >= 2:
        return [{"name": "multiply", "args": {"a": numbers[0], "b": numbers[1]}}]
    return []

def call_llm(prompt: str, functions: List[Dict[str, Any]], use_real: bool = False):
    if not use_real:
        return stub_plan(prompt, functions)
    token = os.getenv("GITHUB_TOKEN")
    
    if not token:
        raise RuntimeError("Set GITHUB_TOKEN or use stub planner.")
    print("I'm here")
    from azure.ai.inference import ChatCompletionsClient
    from azure.core.credentials import AzureKeyCredential
    client = ChatCompletionsClient("https://models.inference.ai.azure.com", AzureKeyCredential(token))
    resp = client.complete(
        model="gpt-4o",
        messages=[{"role": "system", "content": "Plan MCP tool calls."},{"role": "user", "content": prompt}],
        tools=functions,
        temperature=0,
        max_tokens=400,
    )
    calls = []
    msg = resp.choices[0].message
    for tc in msg.tool_calls or []:
        args = tc.function.arguments
        args_json = json.loads(args) if isinstance(args, str) else args
        calls.append({"name": tc.function.name, "args": args_json})
    return calls


In [29]:
async def ex5_run(prompt: str = "Add 2 to 20"):
    params = StdioServerParameters(command="mcp", args=["run", "server.py"]) #To-Do: Create StdioServerParameters with command to run server.py
    async with stdio_client(params, errlog=True) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()
            tools = await session.list_tools()
            functions = [convert_to_llm_tool(t) for t in tools.tools] #To-Do: convert tools to llm tool format
            calls = call_llm(prompt=prompt, functions=functions, use_real=USE_REAL_LLM) #To-Do: get tool calls from call_llm
            print("tool_calls:", calls)
            for call in calls:
                result = await session.call_tool(call["name"], arguments=call["args"])
                print("result:", [getattr(c, "text", str(c)) for c in result.content])


In [31]:
try:
    await ex5_run("Add 2 to 20")
    await ex5_run("multiply 2 and 33")
except* Exception as eg:
    for e in eg.exceptions:
        import traceback; traceback.print_exception(e)

[07/22/26 21:36:18] INFO     Processing request of type            server.py:733
                             ListToolsRequest                                   
I'm here
tool_calls: [{'name': 'add', 'args': {'a': 2, 'b': 20}}]
[07/22/26 21:36:20] INFO     Processing request of type            server.py:733
                             CallToolRequest                                    
result: ['22']
[07/22/26 21:36:21] INFO     Processing request of type            server.py:733
                             ListToolsRequest                                   
I'm here
tool_calls: [{'name': 'multiply', 'args': {'a': 2, 'b': 33}}]
[07/22/26 21:36:22] INFO     Processing request of type            server.py:733
                             CallToolRequest                                    
result: ['66.0']


## Optional - add multiply(a, b) and rerun